# 08 — Perfilado de gustos + asignación de contramedidas

**Bloque 2 (gustos), v1 — 2026-05-28**

Construye sobre los 6 arquetipos del Nivel 1 un sistema de personalización basado en 7 ejes de gusto y un catálogo de 12 contramedidas. Reemplaza al Nivel 2 (HDBSCAN) como capa operacional.

Diseño canónico: [`03_modelo_gustos/archetypes.yaml`](../archetypes.yaml). Lógica reutilizable: [`scripts/gustos/perfilado_gustos.py`](../../scripts/gustos/perfilado_gustos.py).

**Inputs**
- `data/data_qc_gustos/master_table_gustos_v3_aggressive.parquet` (114.412 × 95)
- `data/data_qc_gustos/two_stage_assignments.parquet` (114.412 × 4)

**Outputs**
- `data/data_qc_gustos/player_taste_profile.parquet` (114.412 × 10)
- `data/data_qc_gustos/countermeasure_assignments.parquet` (114.412 × 12)
- `data/data_qc_gustos/_tercile_thresholds.json` (umbrales reproducibles)

## 1. Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from scripts.gustos.perfilado_gustos import (
    build_profile,
    assign_countermeasures,
    COUNTERMEASURES,
    OUT_PROFILE,
    OUT_CM,
    OUT_THRESHOLDS,
)
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Construir el perfil de gustos (7 ejes)

Eje | Fuente | Tipo
---|---|---
clase_favorita | `char_class_main` | universal
estilo_build | `items_attack_defense_ratio` (terciles) | universal
monetizacion | `iap_trial_only`, `iap_is_payer`, `reward_has_ad` | universal
perfil_enhance | `items_max_enhance_level`, `pct_items_high_enhance` | universal
perfil_coleccion | `items_redundancy_ratio`, `coll_total_items` (p75) | universal
pvp_perfil | `fights_pct_pvp`, `fights_pct_won` | Tier 2
perfil_oro | `currency_pct_inflow`, `currency_pct_outflow` | Tier 2

In [2]:
profile, thresholds = build_profile()

print(f'Sample total: {len(profile):,}')
print(f'has_tier2=1:  {(profile["has_tier2"]==1).sum():,}')
print()
print('Umbrales calculados sobre el sample:')
print(json.dumps(thresholds, indent=2, ensure_ascii=False))

Sample total: 114,412
has_tier2=1:  21,599

Umbrales calculados sobre el sample:
{
  "estilo_build": {
    "feature": "items_attack_defense_ratio",
    "t33": 1.0369783412572637,
    "t67": 1.1703931334110416,
    "method": "terciles globales sobre el sample de gustos (N=114.412)"
  },
  "perfil_coleccion": {
    "feature": "coll_total_items",
    "p75": 42.0,
    "method": "percentil 75 global sobre el sample de gustos"
  },
  "perfil_enhance": {
    "rule": "items_max_enhance_level > 2 OR pct_items_high_enhance > 0",
    "rationale": "items_max_enhance_level mediana = 2; pct_items_high_enhance mediana = 0"
  },
  "monetizacion": {
    "prelacion": [
      "trial_no_convertido",
      "pagador",
      "no_pagador_ads",
      "no_pagador"
    ]
  },
  "pvp_perfil": {
    "rule": "pvp_frustrado si fights_pct_pvp > 0.3 AND fights_pct_won < 0.3 (con has_tier2=1); pve_focus en el resto del Tier 2; null si has_tier2=0"
  },
  "perfil_oro": {
    "rule": "acumulador si currency_pct_inflow > 

In [3]:
print('Marginales de los 7 ejes:')
for col in ['clase_favorita', 'estilo_build', 'monetizacion',
            'perfil_enhance', 'perfil_coleccion', 'pvp_perfil', 'perfil_oro']:
    print(f'\n  {col}:')
    vc = profile[col].value_counts(dropna=False)
    for k, v in vc.items():
        print(f'    {k:<25} {v:>7,}  ({v/len(profile)*100:>5.2f}%)')

Marginales de los 7 ejes:

  clase_favorita:
    Asesino                    45,461  (39.73%)
    Caballero                  43,040  (37.62%)
    Campeón                    25,911  (22.65%)

  estilo_build:
    agresivo                   38,138  (33.33%)
    tanque                     38,138  (33.33%)
    equilibrado                38,136  (33.33%)

  monetizacion:
    no_pagador                 96,245  (84.12%)
    no_pagador_ads             15,027  (13.13%)
    pagador                     1,984  ( 1.73%)
    trial_no_convertido         1,156  ( 1.01%)

  perfil_enhance:
    no_inversor                61,812  (54.03%)
    inversor                   52,600  (45.97%)

  perfil_coleccion:
    funcional                  78,343  (68.47%)
    coleccionista              36,069  (31.53%)

  pvp_perfil:
    null                       92,813  (81.12%)
    pve_focus                  20,093  (17.56%)
    pvp_frustrado               1,506  ( 1.32%)

  perfil_oro:
    null                       92,8

In [4]:
# Persistir el perfil y los umbrales
OUT_THRESHOLDS.write_text(json.dumps(thresholds, indent=2, ensure_ascii=False))
profile.to_parquet(OUT_PROFILE, index=False)
print(f'[write] {OUT_PROFILE}  ({profile.shape})')
print(f'[write] {OUT_THRESHOLDS}')

[write] /Users/jezquerro/Documents/tfg/data/data_qc_gustos/player_taste_profile.parquet  ((114412, 10))
[write] /Users/jezquerro/Documents/tfg/data/data_qc_gustos/_tercile_thresholds.json


## 3. Asignar contramedida primaria por reglas de prioridad

Orden de prioridad (más → menos específico), **v1.1 (2026-05-28)**:

1. **CM03** reconversion_trial — trial_no_convertido
2. **CM01** oferta_premium_clase — Hardcore End-Game + pagador
3. **CM02** oferta_premium_dirigida — pagador (resto)
4. **CM10** oferta_conversion_oro — Tier 2 acumulador
5. **CM11** ajuste_pvp — Tier 2 pvp_frustrado
6. **CM12** reto_guiado_clase — Recién Llegado *(subido sobre los gustos: el onboarding manda)*
7. **CM04** skin_clase — Veteranos no pagador
8. **CM06** reto_coleccion — coleccionista + Establecido/Veterano
9. **CM05** evento_enhance — perfil_enhance=inversor
10. **CM07** drop_build — agresivo/tanque + arquetipo activo
11. **CM08** recompensa_ad — no_pagador_ads
12. **CM09** regalo_moneda — Casual Dormido + no_pagador

**Fallback final**: cualquier jugador que no encaje en ninguna regla recibe **CM04** (skin_clase) usando su `clase_favorita`. Esto cierra el gap a 0 — cobertura 100 %.

In [5]:
cm = assign_countermeasures(profile)
cm.to_parquet(OUT_CM, index=False)
print(f'[write] {OUT_CM}  ({cm.shape})')
print()
cm.head().to_string()

[write] /Users/jezquerro/Documents/tfg/data/data_qc_gustos/countermeasure_assignments.parquet  ((114412, 12))



'                    user_id            arquetipo_n1  has_tier2 clase_favorita estilo_build monetizacion perfil_enhance perfil_coleccion pvp_perfil perfil_oro contramedida_primaria_cod contramedida_primaria_label\n0  5f592ddf584aed226e0e835b       Hardcore End-Game          0      Caballero     agresivo      pagador       inversor    coleccionista       null       null                      CM01        oferta_premium_clase\n1  5fd5db71fb368a010b508c24       Veterano Inversor          0        Asesino     agresivo   no_pagador       inversor    coleccionista       null       null                      CM04                  skin_clase\n2  6026fead8844bc1c56007a21          Casual Dormido          0        Campeón       tanque   no_pagador       inversor        funcional       null       null                      CM05              evento_enhance\n3  6193e4a028cfa3039c109ccd       Hardcore End-Game          0        Campeón       tanque      pagador       inversor    coleccionista       null 

## 4. Distribución final de contramedidas

In [6]:
dist = (
    cm.groupby(['contramedida_primaria_cod', 'contramedida_primaria_label'])
    .size().reset_index(name='n')
    .sort_values('n', ascending=False)
)
dist['pct'] = dist['n'] / len(cm) * 100
dist['flag_lt500'] = dist['n'] < 500
dist

,contramedida_primaria_cod,contramedida_primaria_label,n,pct,flag_lt500
8,CM09,regalo_moneda,43566,38.078174,False
4,CM05,evento_enhance,25879,22.619131,False
3,CM04,skin_clase,11891,10.393141,False
9,CM10,oferta_conversion_oro,10832,9.467538,False
5,CM06,reto_coleccion,8485,7.416180,False
7,CM08,recompensa_ad,4048,3.538090,False
6,CM07,drop_build,3269,2.857218,False
11,CM12,reto_guiado_clase,2704,2.363388,False
2,CM03,reconversion_trial,1156,1.010384,False
1,CM02,oferta_premium_dirigida,1004,0.877530,False


In [7]:
# Cruce contramedida × arquetipo (composición por arquetipo)
ct = pd.crosstab(cm['arquetipo_n1'], cm['contramedida_primaria_cod'], margins=True)
ct

contramedida_primaria_cod,CM01,CM02,CM03,CM04,CM05,CM06,CM07,CM08,CM09,CM10,CM11,CM12,All
arquetipo_n1,,,,,,,,,,,,,
Casual Dormido,0,169,220,0,18158,0,0,3970,43566,1220,5,0,67308
Hardcore End-Game,980,0,369,11,4065,0,51,32,0,365,116,0,5989
Jugador Establecido Activo,0,290,207,781,3656,8485,3218,46,0,4097,151,0,20931
Recién Llegado Explorador,0,185,126,0,0,0,0,0,0,4234,243,2704,7492
Veterano Especializado,0,168,50,4663,0,0,0,0,0,389,21,0,5291
Veterano Inversor,0,192,184,6436,0,0,0,0,0,527,62,0,7401
All,980,1004,1156,11891,25879,8485,3269,4048,43566,10832,598,2704,114412


## 5. Cobertura final

Tras los ajustes v1.1 (CM12 priorizado + fallback CM04), el catálogo cubre **100 %** del sample y no hay `CM00 sin_asignar`. La regla 7 (CM04) absorbe ahora dos perfiles distintos:

- **Veteranos no pagador** (regla canónica): los `Veterano Especializado` / `Veterano Inversor` con `monetizacion != pagador`.
- **Fallback** (regla final): cualquier jugador sin match en las 12 reglas anteriores — perfil "activo sin distintivo" en arquetipos no cubiertos.

In [8]:
fallback_breakdown = cm.groupby('arquetipo_n1')['contramedida_primaria_cod'].apply(
    lambda s: (s == 'CM04').sum()
)
veteranos = {'Veterano Especializado', 'Veterano Inversor'}
mask_canonica = (cm['contramedida_primaria_cod'] == 'CM04') & cm['arquetipo_n1'].isin(veteranos)
mask_fallback = (cm['contramedida_primaria_cod'] == 'CM04') & ~cm['arquetipo_n1'].isin(veteranos)
print(f'CM04 total: {int((cm["contramedida_primaria_cod"]=="CM04").sum()):,}')
print(f'  Veteranos no_pagador (regla 7): {int(mask_canonica.sum()):,}')
print(f'  Fallback (resto):               {int(mask_fallback.sum()):,}')
print()
print(f'sin_asignar (CM00): {int((cm["contramedida_primaria_cod"]=="CM00").sum()):,}')
print(f'cobertura: {((cm["contramedida_primaria_cod"]!="CM00").sum() / len(cm) * 100):.4f}%')

CM04 total: 11,891
  Veteranos no_pagador (regla 7): 11,099
  Fallback (resto):               792

sin_asignar (CM00): 0
cobertura: 100.0000%
